# Multi-turn conversations

Pass a `session_id` to stack follow-ups. TalkDB rewrites each follow-up into a standalone question before running it through the normal pipeline — you can inspect the rewritten form in the session log.

In [1]:
import os
from pathlib import Path

# Locate project root (so relative paths data/, packages/, .env resolve regardless of
# where the notebook was launched from).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('could not locate project root (no pyproject.toml found upward)')
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()
if not (os.environ.get('OPENAI_API_KEY') or os.environ.get('ANTHROPIC_API_KEY')):
    raise RuntimeError('Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your .env before running this notebook.')
print('project root:', PROJECT_ROOT)

project root: /Users/nitingupta/Desktop/talkdb


In [2]:
from talkdb.config.settings import get_settings
from talkdb.core.engine import Engine

engine = Engine(get_settings())
engine.build_index()
print('engine ready')

engine ready


## Turn 1 — initial question

In [3]:
import uuid

session_id = f'demo_{uuid.uuid4().hex[:8]}'

r1 = await engine.ask('What is total revenue by month?', session_id=session_id)
print('SQL:')
print(r1.sql)
print(f'\n{r1.row_count} rows, confidence {r1.confidence}')

SQL:
SELECT strftime('%Y-%m', o.created_at) AS month, SUM(o.total_amount) AS revenue FROM orders o WHERE o.status = 'completed' GROUP BY month ORDER BY month

13 rows, confidence 74


## Turn 2 — follow-up (`just Q4`)

'Q4' has no meaning on its own — it makes sense only relative to turn 1.

In [4]:
r2 = await engine.follow_up('just Q4', session_id=session_id)
print('SQL:')
print(r2.sql)
print(f'\n{r2.row_count} rows')

SQL:
SELECT strftime('%Y-%m', o.created_at) AS month, SUM(o.total_amount) AS revenue 
FROM orders o 
WHERE o.status = 'completed' AND o.created_at >= '2023-10-01' AND o.created_at < '2024-01-01' 
GROUP BY month 
ORDER BY month

0 rows


## Turn 3 — `break that down by payment method`

In [5]:
r3 = await engine.follow_up('break that down by payment method', session_id=session_id)
print('SQL:')
print(r3.sql)
print()
for row in r3.results[:8]:
    print(' ', row)

SQL:
SELECT strftime('%Y-%m', o.created_at) AS month, SUM(o.total_amount) AS revenue 
FROM orders o 
WHERE o.status = 'completed' AND o.created_at >= '2023-10-01' AND o.created_at < '2024-01-01' 
GROUP BY month 
ORDER BY month



## Inspect the session — see what each turn was rewritten to

In [6]:
state = engine.get_session(session_id)
for turn in state['turns']:
    print(f"Turn {turn['turn_number']}: {turn['question']}")
    if turn['question'] != turn['rewritten_question']:
        print(f"  ↳ rewritten: {turn['rewritten_question']}")
    print(f"  SQL: {turn['sql'][:100]}{'...' if len(turn['sql']) > 100 else ''}")
    print()

Turn 1: What is total revenue by month?
  SQL: SELECT strftime('%Y-%m', o.created_at) AS month, SUM(o.total_amount) AS revenue FROM orders o WHERE ...

Turn 2: just Q4
  ↳ rewritten: What is the total revenue by month for the fourth quarter (Q4)?
  SQL: SELECT strftime('%Y-%m', o.created_at) AS month, SUM(o.total_amount) AS revenue 
FROM orders o 
WHER...

Turn 3: break that down by payment method
  ↳ rewritten: What is the total revenue by month for the fourth quarter (Q4) broken down by payment method?
  SQL: SELECT strftime('%Y-%m', o.created_at) AS month, SUM(o.total_amount) AS revenue 
FROM orders o 
WHER...

